In [ ]:
import numpy as np

onehots = {}
onehots['cat'] = np.array([1,0,0,0])
onehots['the'] = np.array([0,1,0,0])
onehots['dog'] = np.array([0,0,1,0])
onehots['sat'] = np.array([0,0,0,1])
sentence = ['the','cat','sat']
x = onehots[sentence[0]] + onehots[sentence[1]] + onehots[sentence[2]]
print("Sent Encoding:" + str(x))

Sent Encoding:[1 1 0 1]


In [ ]:
f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()
f = open('labels.txt')
raw_labels = f.readlines()
f.close()
tokens = []
for review in raw_reviews:
  words = review.split(" ")
  tokens.append(set(words))
print(tokens)

[{'this', 'and', 'fantastic', 'movie', 'inspiring\n', 'was'}, {'this', 'hated', 'terrible\n', 'i', 'film', 'it', 'was'}, {'and', 'loved', 'absolutely', 'story\n', 'acting', 'the'}, {'predictable\n', 'boring', 'and', 'plot', 'was', 'the'}, {'wonderful', 'and', 'great', 'a', 'experience\n', 'what'}, {'this', 'movie', 'worst', 'ever\n', 'was', 'the'}]


In [ ]:
vocab = set()
for sent in tokens:
  for word in sent:
    if (len(word) > 0):
      vocab.add(word)

vocab = list(vocab)
print(len(vocab))

26


In [ ]:
word2index = {}
for i, word in enumerate(vocab):
  word2index[word] = i
print(word2index)

{'this': 0, 'hated': 1, 'predictable\n': 2, 'a': 3, 'fantastic': 4, 'absolutely': 5, 'worst': 6, 'what': 7, 'it': 8, 'the': 9, 'boring': 10, 'and': 11, 'plot': 12, 'great': 13, 'loved': 14, 'inspiring\n': 15, 'was': 16, 'wonderful': 17, 'terrible\n': 18, 'i': 19, 'story\n': 20, 'movie': 21, 'experience\n': 22, 'film': 23, 'ever\n': 24, 'acting': 25}


In [ ]:
input_dataset = list()
for sent in tokens:
  sent_indices = list()
  for word in sent:
     try:
       sent_indices.append(word2index[word])
     except:
      ""
  input_dataset.append(list(set(sent_indices)))

print(input_dataset)

[[0, 4, 11, 15, 16, 21], [0, 1, 8, 16, 18, 19, 23], [5, 9, 11, 14, 20, 25], [2, 9, 10, 11, 12, 16], [3, 7, 11, 13, 17, 22], [0, 6, 9, 16, 21, 24]]


In [ ]:
target_dataset = list()
for label in raw_labels:
  if label == 'positive\n':
    target_dataset.append(1)
  else:
    target_dataset.append(0)

print(target_dataset)

[1, 0, 1, 0, 1, 0]


In [ ]:
np.random.seed(42)

def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [ ]:
alpha, iterations = (0.01, 10)
hidden_size = 100

In [ ]:
weights_0_1 = np.random.random((len(vocab), hidden_size))
weights_1_2 = np.random.random((hidden_size, 1))

for iter in range(iterations):
  for i in range(len(input_dataset)):
    x, y = (input_dataset[i], target_dataset[i])
    layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
    layer_2 = sigmoid(np.dot(layer_1, weights_1_2))
    layer_2_delta = layer_2 - y
    layer_1_delta = layer_2_delta.dot(weights_1_2.T)
    weights_0_1[x] -= layer_1_delta * alpha
    layer_1 = layer_1.reshape(hidden_size, 1)
    layer_2_delta = layer_2_delta.reshape(1, 1)
    weights_1_2 -= np.dot(layer_1, layer_2_delta) * alpha
    correct, total = (0, 0)
    for i in range(len(input_dataset)):
      x = input_dataset[i]
      y = target_dataset[i]
      layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
      layer_2 = sigmoid(np.dot(layer_1, weights_1_2))
      if (np.abs(layer_2 - y) < 0.5):
        correct += 1
      total += 1
print(correct / total)

0.5


In [ ]:
from collections import Counter
import math
def similar(target='fantastic'):
  target_index = word2index[target]
  scores = Counter()
  for word, index in word2index.items():
    raw_difference = weights_0_1[index] - (weights_0_1[target_index])
    squared_difference = raw_difference * raw_difference
    scores[word] = -math.sqrt(sum(squared_difference))
  return scores.most_common(10)
print(similar('fantastic'))
print(similar('boring'))

[('fantastic', -0.0), ('boring', -3.829433685494326), ('the', -3.9346707318903147), ('worst', -3.9814705947597524), ('i', -4.026249409452766), ('inspiring\n', -4.1378949226386394), ('a', -4.166987772577432), ('acting', -4.213122735129478), ('absolutely', -4.214402458940089), ('experience\n', -4.232222838315802)]
[('boring', -0.0), ('movie', -3.602759682217969), ('terrible\n', -3.6186690414317875), ('the', -3.6555568950409643), ('i', -3.772768847200205), ('and', -3.786220004253691), ('fantastic', -3.829433685494326), ('worst', -3.854440412626511), ('ever\n', -3.8795067557113576), ('absolutely', -4.04282997144749)]


In [ ]:
import random

np.random.seed(42)
random.seed(42)

In [ ]:
concatenated = []
input_dataset = []
for sent in tokens:
  sent_indices = []
  for word in sent:
    try:
      sent_indices.append(word2index[word])
      concatenated.append(word2index[word])
    except:
      ""
  input_dataset.append(sent_indices)
concatenated = np.array(concatenated)
random.shuffle(input_dataset)

In [65]:
alpha, iterations = (0.05, 100)
hidden_size, window, negative = (50, 2, 5)

weights_0_1 = np.random.rand(len(vocab), hidden_size)
weights_1_2 = np.random.rand(len(vocab), hidden_size)
layer_2_target = np.zeros(negative + 1)
layer_2_target[0] = 1

for rev_i, review in enumerate(input_dataset * iterations):
  for target_i in range(len(review)):
    target_samples = [review[target_i]] + list(concatenated[(np.random.rand(negative) * len(concatenated)).astype('int').tolist()])
    left_context = review[max(0, target_i - window):target_i]
    right_context = review[target_i + 1:min(len(review), target_i + window)]